This code can be executed on Kaggle.

In [6]:
!pip install -q evaluate rouge_score sacrebleu nltk

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 14.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.8.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platfor

In [7]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
WANDB_API_KEY = user_secrets.get_secret("WANDB_API_KEY")

In [8]:
from huggingface_hub import login

login(token=HF_TOKEN)

In [9]:
import os
os.environ["WANDB_API_KEY"] = WANDB_API_KEY
os.environ["TOKENIZERS_PARALLELISM"] = "true"

import wandb

In [10]:
!rm -rf /kaggle/working/event-planned-story-gen
!git clone https://github.com/abirmondal/event-planned-story-gen.git

Cloning into 'event-planned-story-gen'...
remote: Enumerating objects: 273, done.
remote: Counting objects: 100% (146/146), done.
remote: Compressing objects: 100% (94/94), done.
remote: Total 273 (delta 69), reused 117 (delta 46), pack-reused 127 (from 1)
Receiving objects: 100% (273/273), 57.13 MiB | 22.41 MiB/s, done.
Resolving deltas: 100% (124/124), done.
Updating files: 100% (44/44), done.


In [11]:
import sys
from pathlib import Path

# Add the parent directory's path to sys.path
# sys.path requires strings, so we convert the Path object
sys.path.append(str("/kaggle/working/event-planned-story-gen"))

In [12]:
import numpy as np
import evaluate
from src.dataset_prep.data_for_train import DataForTrain
import config.event_special_tokens as event_special_tokens
from src.utils.event_evals import calculate_metrics_for_events
from transformers import (
    BartForConditionalGeneration,
    BartTokenizerFast,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)

2025-09-03 14:39:14.755342: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756910354.951906      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756910355.007744      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [13]:
event_filename_suffix = '_event.source_new'
data_prep = DataForTrain(
    event_filename_suffix=event_filename_suffix,
    data_types=['train', 'val', 'test']
)

dataset_dict = data_prep.get_data_for_lc_to_event()

Processing train data:   0%|          | 0/88344 [00:00<?, ?it/s]

Processing val data:   0%|          | 0/4908 [00:00<?, ?it/s]

Processing test data:   0%|          | 0/4909 [00:00<?, ?it/s]

In [14]:
for split_name, dataset in dataset_dict.items():
    dataset_dict[split_name] = dataset.shuffle(seed=42)

In [15]:
# Run this code to test the training piepline

# for split_name, dataset in dataset_dict.items():
#     dataset_dict[split_name] =  dataset.select(range(100))

In [16]:
model_name = "facebook/bart-base"
tokenizer = BartTokenizerFast.from_pretrained(model_name)

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

In [17]:
special_tokens_dict = {
    "additional_special_tokens": [
        event_special_tokens.EVENT_START,
        event_special_tokens.EVENT_SEPERATOR,
        event_special_tokens.EVENT_END
    ]
}

tokenizer.add_special_tokens(special_tokens_dict)

3

In [18]:
model = BartForConditionalGeneration.from_pretrained(model_name)
model.resize_token_embeddings(len(tokenizer))

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


BartScaledWordEmbedding(50268, 768, padding_idx=1)

In [19]:
max_source_length = 128
max_target_length = 32

def preprocess(example):
    inputs = tokenizer(
        example["source"],
        max_length=max_source_length,
        truncation=True,
        padding="max_length"
    )
    targets = tokenizer(
        example["event"],
        max_length=max_target_length,
        truncation=True,
        padding="max_length"
    )
    inputs["labels"] = targets["input_ids"]
    return inputs

tokenized_datasets = dataset_dict.map(
    preprocess,
    batched=True,
    remove_columns=dataset_dict["train"].column_names,
)

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [20]:
wandb_project = "lc-to-event-BART"
hf_repo_id = "abirmondalind/lc-to-event-BART"

In [21]:
wandb.init(
    project=wandb_project,
    config={
        "model_name": model_name,
        "dataset": "ROCStories",
        "max_source_length": max_source_length,
        "max_target_length": max_target_length,
        "event_filename_suffix": event_filename_suffix,
    }
)

In [22]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    run_name="lc-to-event-BART-v2",
    
    num_train_epochs=5,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    learning_rate=1e-4,
    adam_epsilon=1e-8,
    label_smoothing_factor=0.0,
    weight_decay=0.01,
    fp16=True,
    seed=42,
    
    # eval_strategy="epoch", # Test the trainng pipeline
    # save_strategy="epoch", # Test the trainng pipeline
    eval_strategy="steps",
    eval_steps=1000,
    save_strategy="steps",
    save_steps=1000,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="rouge2",
    greater_is_better=True,
    predict_with_generate=True,
    generation_num_beams=4,
    dataloader_num_workers=2,
      
    logging_steps=250,
    report_to="wandb",
    push_to_hub=True,
    hub_model_id=hf_repo_id,
    hub_strategy="every_save",
)

In [23]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [24]:
compute_metrics_eval = calculate_metrics_for_events(tokenizer)

In [25]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["val"],
    data_collator=data_collator,
    compute_metrics=compute_metrics_eval
)

In [26]:
trainer.train()

wandb: Currently logged in as: abirmondalind to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum,Bleu,Gen Len
1,No log,11.820024,0.029704,0.000000,0.029419,0.029470,0.000000,20.590000
2,No log,11.049761,0.010902,0.000000,0.011183,0.010842,0.000000,21.000000
3,No log,10.451502,0.004286,0.000000,0.004762,0.004762,0.000000,21.000000
4,No log,10.048681,0.003333,0.000000,0.003333,0.003333,0.000000,21.000000
5,No log,9.849288,0.003333,0.000000,0.003333,0.003333,0.000000,21.000000


/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:3465: UserWarning: Moving the following attributes in the config to the generation config: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tens

TrainOutput(global_step=5, training_loss=11.822544860839844, metrics={'train_runtime': 52.4731, 'train_samples_per_second': 9.529, 'train_steps_per_second': 0.095, 'total_flos': 38108528640000.0, 'train_loss': 11.822544860839844, 'epoch': 5.0})

In [27]:
raw_predictions = trainer.predict(tokenized_datasets["test"])

/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


In [28]:
cal_metrices_events_comp_fn = calculate_metrics_for_events(
    tokenizer,
    metrics_prefix="test",
    save_preds=True,
    save_preds_file_name=wandb.run.name+"_preds.csv"
)
metrics = cal_metrices_events_comp_fn((raw_predictions.predictions, raw_predictions.label_ids))
wandb.log(metrics)

In [ ]:
trainer.push_to_hub()